[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/ofermend/hands-on-rag/blob/main/chapter4/redaction.ipynb)

# Chapter 4: Data Anonymization for RAG

When building RAG systems over sensitive documents, personally identifiable information (PII) like names, phone numbers, and addresses can leak into generated responses. Data anonymization ensures that PII is detected and redacted before documents enter the retrieval pipeline — or before responses reach end users.

This notebook demonstrates entity-aware redaction using Microsoft's Presidio library, which replaces detected PII with typed placeholders (e.g., `<PERSON>`, `<PHONE_NUMBER>`) to preserve semantic structure.

**What you'll learn:**
- Detect PII entities (names, phone numbers) using Presidio's analyzer
- Apply typed masking to replace sensitive data with category labels
- Preserve document readability while removing identifying information

**Prerequisites:** `pip install presidio-analyzer presidio-anonymizer`

## Setup

We import Presidio's analyzer (for PII detection) and anonymizer (for replacement), which together form the redaction pipeline.

In [1]:
from presidio_analyzer import AnalyzerEngine
from presidio_anonymizer import AnonymizerEngine
from presidio_anonymizer.entities import OperatorConfig

## Entity-Aware Redaction

This function detects PII entities in text and replaces each with a typed placeholder like `<PERSON>` or `<PHONE_NUMBER>`, preserving readability while removing identifying information.

In [2]:
def entity_aware_redaction(text):
    """
    Implements the 'Typed Masking' strategy.
    Replaces sensitive data with its category (e.g., [PERSON])
    """
    analyzer = AnalyzerEngine()
    anonymizer = AnonymizerEngine()

    # 1. Detect PII entities
    results = analyzer.analyze(text=text, entities=["PERSON", "PHONE_NUMBER"], language='en')

    # 2. Replace with Entity Type (preserving semantic structure)
    anonymized_result = anonymizer.anonymize(
        text=text,
        analyzer_results=results,
        operators={
            "PERSON": OperatorConfig("replace", {"new_value": "<PERSON>"}),
            "PHONE_NUMBER": OperatorConfig("replace", {"new_value": "<PHONE_NUMBER>"}),
        }
    )
    
    return anonymized_result.text



## Testing the Redaction

We pass a sentence containing a name and phone number through the redaction function to verify that both entity types are correctly detected and masked.

In [3]:
input = "Dr. Bao called 123-555-1122."
output = entity_aware_redaction(input)
print(output)


Dr. <PERSON> called <PHONE_NUMBER>.
